In [5]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import pickle
import pandas as pd

In [ ]:
def get_data(source_dir: str, target_af=1.0):
    """核心解析逻辑（保持不变）"""
    res = []
    if not os.path.exists(source_dir):
        return res
    
    for name in os.listdir(source_dir):
        if not os.path.isdir(os.path.join(source_dir, name)) and '-' in name:
            try:
                clean_name = name.rsplit('.', 1)[0]
                parts = clean_name.split('-')
                if len(parts) < 6: continue
                
                algo, ub, d, budget, af, task_name = parts[:6]

                if float(af) == target_af:
                    if algo == 'EfficientBFS' and ub == 'ub0':
                        continue
                    
                    file_path = os.path.join(source_dir, name)
                    with open(file_path, "rb") as rd:
                        kv_data = pickle.load(rd)

                    res.append({
                        'task': task_name,
                        'algorithm': algo,
                        'budget': float(eval(budget)),
                        'heuristic': ub,
                        'sorting': d,
                        'node_count': kv_data.get('node_count', 0),
                        'open_list_count': kv_data.get('open_list_count', 0),
                        'computational_time': kv_data.get('time', 0),
                        'f_S': kv_data.get('f(S)', 0)
                    })
            except Exception:
                continue
    return res

# --- 1. 配置区域 ---
# 定义需要处理的任务及其对应的 n
TASKS_CONFIG = [
    {"name": "facebook", "n": 1000},
    {"name": "youtube",  "n": 1000}
]

# 定义需要包含的 Seeds
SEEDS = [0, 1, 2]

TARGET_AF = 0.95
ARCHIVE_DIR = "../result/archive-92"
RESULT_DIR = "./result_92"
ALGS_TO_KEEP = ['EfficientBFS', 'Efficient', 'ILP', 'BFSTC']
BUDGET_RANGE = range(6, 16)

# --- 2. 嵌套循环执行 ---
os.makedirs(RESULT_DIR, exist_ok=True)
all_data_frames = []

for config in TASKS_CONFIG:
    for s in SEEDS:
        task_name = config["name"]
        n_val = config["n"]
        
        # 动态构造路径: ../result/archive-92/facebook/1000/0
        source_path = os.path.join(ARCHIVE_DIR, task_name, str(n_val), str(s))
        
        print(f"Loading: {task_name} | n={n_val} | seed={s}...")
        
        task_res = get_data(source_dir=source_path, target_af=TARGET_AF)
        
        if task_res:
            temp_df = pd.DataFrame(task_res)
            temp_df['seed'] = s  # 关键：添加 seed 列作为维度
            temp_df['n'] = n_val
            all_data_frames.append(temp_df)

# --- 3. 合并与过滤 ---
if not all_data_frames:
    print("未找到任何匹配数据，请检查路径。")
else:
    df = pd.concat(all_data_frames, ignore_index=True)

    # 逻辑过滤
    cond1 = (df['algorithm'] == 'BFSTC') & (df['heuristic'] == 'ub0')
    cond2 = (df['algorithm'] != 'BFSTC')
    filtered_df = df[(df['algorithm'].isin(ALGS_TO_KEEP)) & (cond1 | cond2)]
    filtered_df = filtered_df[filtered_df['budget'].isin(BUDGET_RANGE)]

    # --- 4. 导出 Excel ---
    output_xlsx = os.path.join(RESULT_DIR, "fb_yt_multi_seeds.xlsx")
    
    # 排序：任务 -> Seed -> 预算 -> 算法
    filtered_df = filtered_df.sort_values(by=['task', 'seed', 'budget', 'algorithm'])
    
    # 导出为 Excel (需要安装 openpyxl: pip install openpyxl)
    filtered_df.to_excel(output_xlsx, index=False, engine='openpyxl')
    
    print("-" * 30)
    print(f"Excel 已生成: {output_xlsx}")
    print(f"总行数: {len(filtered_df)}")

------------------------------
修正版数据已生成：
Excel 路径: ./result_92\fb_yt_multi_seeds.xlsx
CSV 路径 (推荐): ./result_92\fb_yt_multi_seeds.csv
